# Data Acquisition Guide
## National Transport Decarbonisation Dashboard for Ireland

This notebook is the **single reference point** for every data source used in the
project. It documents where each dataset comes from, how to download or refresh
it, and how to name and store it so the rest of the pipeline picks it up
automatically.

> **Run this notebook first** whenever you need to refresh the data. After
> downloading, run `01_data_preprocessing_and_cleaning.ipynb` to rebuild the
> processed outputs.

### Principle: single source of truth
All raw data must come from the official CSO PxStat / data.gov.ie portals. No
ad-hoc alternative estimates or copy-pasted figures. This ensures every KPI and
forecast in the dashboard is fully traceable back to a citable official source.

## 1. Data source catalogue

The table below documents all eight datasets used in the project, plus the
supplementary NaPTAN spatial layer.

In [9]:
import os
from pathlib import Path

# Walk up from wherever Jupyter started until we find the repo root
def _find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "requirements.txt").exists():
            return candidate
    return Path.cwd()

REPO_ROOT = Path(os.environ.get("TDD_REPO_ROOT", str(_find_repo_root())))
RAW_DIR   = Path(os.environ.get("TDD_RAW_DIR",   str(REPO_ROOT / "data" / "raw")))
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
DOCS_DIR      = REPO_ROOT / "docs"

print("Repo root :", REPO_ROOT)
print("Raw dir   :", RAW_DIR)
print("Exists?   :", RAW_DIR.exists())

Repo root : c:\Users\srb10\Desktop\MSc BA Project groups [Semester - 2]\IS6611\New folder\Transport_Decarbonisation_Dashboard_IE
Raw dir   : c:\Users\srb10\Desktop\MSc BA Project groups [Semester - 2]\IS6611\New folder\Transport_Decarbonisation_Dashboard_IE\data\raw
Exists?   : True


In [10]:
import os, glob
from pathlib import Path
from datetime import datetime

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
RAW_DIR = Path(os.environ.get("TDD_RAW_DIR", REPO_ROOT / "data" / "raw"))

CATALOGUE = [
    {
        "id":          "TOA11",
        "description": "Luas Passenger Numbers (Red & Green line, monthly)",
        "source":      "CSO PxStat / data.gov.ie",
        "time_range":  "2018–2025",
        "frequency":   "Monthly (released ~6 weeks after reference month)",
        "url":         "https://data.cso.ie/table/TOA11",
        "use":         "Public Transport Usage Index (Luas journeys per capita)",
    },
    {
        "id":          "PEA01",
        "description": "Population Estimates by Age Group and Sex (April reference)",
        "source":      "CSO PxStat",
        "time_range":  "2018–2025",
        "frequency":   "Annual (released ~August each year)",
        "url":         "https://data.cso.ie/table/PEA01",
        "use":         "Denominator for all per-capita KPIs",
    },
    {
        "id":          "THA25",
        "description": "Public Transport Passenger Journeys by Mode (weekly, bus & rail, excl. Luas)",
        "source":      "CSO PxStat / data.gov.ie",
        "time_range":  "2019–2025",
        "frequency":   "Weekly (released ~2 weeks in arrears)",
        "url":         "https://data.cso.ie/table/THA25",
        "use":         "Public Transport Usage Index (bus + rail journeys)",
    },
    {
        "id":          "THA18",
        "description": "Road Traffic – Private Cars by Engine Capacity, Fuel & County",
        "source":      "CSO PxStat",
        "time_range":  "2018–2023",
        "frequency":   "Annual (released ~Q2 following reference year)",
        "url":         "https://data.cso.ie/table/THA18",
        "use":         "Car Dependency Index; private-car km for intensity cross-check",
    },
    {
        "id":          "THA17",
        "description": "Road Traffic Volumes – All Vehicles by Fuel Type & County",
        "source":      "CSO PxStat",
        "time_range":  "2018–2023",
        "frequency":   "Annual (released ~Q2 following reference year)",
        "url":         "https://data.cso.ie/table/THA17",
        "use":         "Transport Intensity Indicator (total vehicle-km per capita)",
    },
    {
        "id":          "TEM12",
        "description": "New Vehicles Licensed for the First Time by Fuel Type (monthly)",
        "source":      "CSO PxStat",
        "time_range":  "2015–2026",
        "frequency":   "Monthly (released within ~2 weeks of month end)",
        "url":         "https://data.cso.ie/table/TEM12",
        "use":         "Fuel Transition analysis (petrol → EV/hybrid share trend)",
    },
    {
        "id":          "TEM22",
        "description": "New & Second-hand Private Cars Licensed (historical 2019–2021)",
        "source":      "CSO PxStat",
        "time_range":  "2019–2021",
        "frequency":   "Superseded by TEM23 — static historical archive",
        "url":         "https://data.cso.ie/table/TEM22",
        "use":         "Private car registration series 2019–2021 (bridged to TEM23)",
    },
    {
        "id":          "TEM23",
        "description": "All Private Cars Licensed for the First Time (new + imported)",
        "source":      "CSO PxStat",
        "time_range":  "2022–2026",
        "frequency":   "Monthly (released within ~2 weeks of month end)",
        "url":         "https://data.cso.ie/table/TEM23",
        "use":         "Private car registration series 2022–2026",
    },
    {
        "id":          "NaPTAN",
        "description": "National Public Transport Access Nodes – Bus Stop Points",
        "source":      "data.gov.ie (National Transport Authority)",
        "time_range":  "Static (updated periodically)",
        "frequency":   "No fixed schedule — check data.gov.ie for updates",
        "url":         "https://data.gov.ie/dataset/national-public-transport-access-nodes-naptan",
        "use":         "Optional map layer showing bus/rail stop coverage",
    },
]

col_w = [7, 62, 20, 12, 10]
header = f"{'Table':<{col_w[0]}}  {'Description':<{col_w[1]}}  {'Source':<{col_w[2]}}  {'Range':<{col_w[3]}}  {'Frequency'}"
sep    = "  ".join("-"*w for w in col_w)
print(header)
print(sep)
for row in CATALOGUE:
    desc = row["description"][:col_w[1]]
    src  = row["source"][:col_w[2]]
    freq = row["frequency"][:col_w[4]] if len(row["frequency"]) > 40 else row["frequency"]
    print(f"{row['id']:<{col_w[0]}}  {desc:<{col_w[1]}}  {src:<{col_w[2]}}  {row['time_range']:<{col_w[3]}}  {freq}")

Table    Description                                                     Source                Range         Frequency
-------  --------------------------------------------------------------  --------------------  ------------  ----------
TOA11    Luas Passenger Numbers (Red & Green line, monthly)              CSO PxStat / data.go  2018–2025     Monthly (r
PEA01    Population Estimates by Age Group and Sex (April reference)     CSO PxStat            2018–2025     Annual (released ~August each year)
THA25    Public Transport Passenger Journeys by Mode (weekly, bus & rai  CSO PxStat / data.go  2019–2025     Weekly (released ~2 weeks in arrears)
THA18    Road Traffic – Private Cars by Engine Capacity, Fuel & County   CSO PxStat            2018–2023     Annual (re
THA17    Road Traffic Volumes – All Vehicles by Fuel Type & County       CSO PxStat            2018–2023     Annual (re
TEM12    New Vehicles Licensed for the First Time by Fuel Type (monthly  CSO PxStat            2015–2026     

## 2. Manual download instructions

This is the **recommended method** for updating the dataset. The CSO PxStat
web interface provides a point-and-click export with full control over filters
and time periods.

### Step-by-step (for each table listed above)

1. Open the table URL from the catalogue (e.g.
   `https://data.cso.ie/table/TEM23`).
2. In the **Filters** panel, ensure the full time range is selected (select
   *all* years/months by default — do not narrow the selection).
3. Click **Download** (top-right of the table view) → select **CSV**.
4. Save the file into `data/raw/` using the naming convention below.

### File naming convention

```
<TABLE_ID>_<YYYYMMDDTHHMMSS>_<STARTYEAR><ENDYEAR>.csv
```

Examples already in `data/raw/`:
```
TEM23_20260528T000533_20222026.csv
TOA11_20260528T000523_20182025.csv
```

The pipeline matches files by **prefix only** (`TEM23*.csv`), so the timestamp
and year suffix are metadata-for-humans, not parsed by code. Newer downloads
with a later timestamp are picked up automatically — old copies can be deleted
or archived.

> **Do not rename the columns.** The pipeline expects the exact CSO column
> headers. If CSO revises a table's structure, update the cleaning cell in
> notebook 01 and add an audit note.

## 3. Programmatic download via the CSO PxStat API

The CSO PxStat API uses the **JSON-RPC 2.0** protocol. No API key is required.
The endpoint supports CSV, JSON-stat and PX format responses.

Use this approach if you need to automate refreshes on a schedule (e.g. in a
GitHub Actions workflow). For ad-hoc updates, the manual method in section 2 is
simpler.

In [11]:
import json, os, re
from datetime import datetime, timezone
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
RAW_DIR = Path(os.environ.get("TDD_RAW_DIR", REPO_ROOT / "data" / "raw"))

API_ENDPOINT = "https://ws.cso.ie/public/api.jsonrpc"

def build_api_payload(table_id: str, fmt: str = "CSV") -> dict:
    """Build a JSON-RPC 2.0 payload to read a full CSO PxStat table as CSV."""
    return {
        "jsonrpc": "2.0",
        "method": "PxStat.Data.Cube_API.ReadDataset",
        "params": {
            "class": "query",
            "id": [],
            "dimension": {},
            "extension": {
                "pivot": None,
                "codes": False,
                "language": {"code": "en"},
                "format": {"type": fmt, "version": "2.0"},
                "matrix": table_id.upper(),
            },
            "version": "2.0",
        },
    }

def download_table(table_id: str, raw_dir: Path = RAW_DIR,
                   timeout: int = 60) -> Path | None:
    """Download a CSO PxStat table to raw_dir as a timestamped CSV.

    Returns the saved Path on success, None on any network error.
    """
    try:
        import requests
        payload = build_api_payload(table_id)
        resp = requests.post(API_ENDPOINT, json=payload,
                             headers={"Content-Type": "application/json"},
                             timeout=timeout)
        resp.raise_for_status()
        data = resp.json()
        if "error" in data:
            print(f"  API error for {table_id}: {data['error']}")
            return None
        csv_text = data["result"]      # CSV content as a string
        ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
        # Guess the year span from the first and last year values in the CSV.
        years = re.findall(r"(20\d{2})", csv_text)
        span  = f"_{min(years)}{max(years)}" if years else ""
        path  = raw_dir / f"{table_id.upper()}_{ts}{span}.csv"
        raw_dir.mkdir(parents=True, exist_ok=True)
        path.write_text(csv_text, encoding="utf-8")
        print(f"  Saved {path.name}  ({len(csv_text):,} chars)")
        return path
    except ImportError:
        print(f"  requests not installed — pip install requests")
    except Exception as e:
        print(f"  Network/API error for {table_id}: {type(e).__name__}: {e}")
    return None

# ── Example: download TEM12 ──────────────────────────────────────────
# Uncomment the line below to run a live download:
#   saved = download_table("TEM12")
#
# To refresh all core tables at once:
#   CORE_TABLES = ["TOA11","PEA01","THA25","THA18","THA17","TEM12","TEM22","TEM23"]
#   for t in CORE_TABLES: download_table(t)

print("API helper defined.")
print(f"Endpoint  : {API_ENDPOINT}")
print(f"Raw dir   : {RAW_DIR}")
print()
print("To run a live download, uncomment the example above.")
print("Requires: pip install requests")

API helper defined.
Endpoint  : https://ws.cso.ie/public/api.jsonrpc
Raw dir   : c:\Users\srb10\Desktop\MSc BA Project groups [Semester - 2]\IS6611\New folder\Transport_Decarbonisation_Dashboard_IE\data\raw

To run a live download, uncomment the example above.
Requires: pip install requests


## 4. Data freshness checker

Run this cell to inspect what is currently in `data/raw/` and whether every
expected table has a recent copy. A file is considered **stale** if it is older
than the number of days listed for its expected refresh cycle.

In [14]:
import os
from pathlib import Path
from datetime import datetime, timezone, timedelta

def _find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "requirements.txt").exists():
            return candidate
    return Path.cwd()

REPO_ROOT = Path(os.environ.get("TDD_REPO_ROOT", str(_find_repo_root())))
RAW_DIR = Path(os.environ.get("TDD_RAW_DIR", str(REPO_ROOT / "data" / "raw")))

EXPECTED = {
    "TOA11": {"label": "Luas Passenger Numbers",             "refresh_days": 90},
    "PEA01": {"label": "Population Estimates",               "refresh_days": 365},
    "THA25": {"label": "PT Journeys (weekly)",               "refresh_days": 30},
    "THA18": {"label": "Private Car Stock",                  "refresh_days": 365},
    "THA17": {"label": "All-Vehicle Stock & KM",             "refresh_days": 365},
    "TEM12": {"label": "New Vehicles by Fuel Type",          "refresh_days": 30},
    "TEM22": {"label": "Private Cars 2019-21 (archive)",     "refresh_days": 9999},
    "TEM23": {"label": "All Private Cars 2022+",             "refresh_days": 30},
}

now  = datetime.now(timezone.utc)
sep  = "-" * 82
print(f"{'Table':<7}  {'Label':<40}  {'File found':<24}  {'Age':>8}  Status")
print(sep)

for tid, meta in EXPECTED.items():
    matches = sorted(RAW_DIR.glob(f"{tid}*.csv"))
    if not matches:
        print(f"{tid:<7}  {meta['label']:<40}  {'NOT FOUND':<24}  {'':>9}  ⚠ MISSING")
        continue
    latest   = matches[-1]
    mtime    = datetime.fromtimestamp(latest.stat().st_mtime, tz=timezone.utc)
    age_days = (now - mtime).days
    age_str  = f"{age_days}d ago"
    stale    = age_days > meta["refresh_days"] and meta["refresh_days"] < 9999
    status   = "⚠ STALE" if stale else "✓ ok"
    fname    = latest.name[:24]
    print(f"{tid:<7}  {meta['label']:<40}  {fname:<24}  {age_str:>8}  {status}")

print(sep)
print(f"Checked {len(EXPECTED)} tables in: {RAW_DIR}")

Table    Label                                     File found                     Age  Status
----------------------------------------------------------------------------------
TOA11    Luas Passenger Numbers                    TOA11.20260528T000523 20    0d ago  ✓ ok
PEA01    Population Estimates                      PEA01.20260528T000539 20    0d ago  ✓ ok
THA25    PT Journeys (weekly)                      THA25.20260528T000529 20    0d ago  ✓ ok
THA18    Private Car Stock                         THA18.20260528T000504 20    0d ago  ✓ ok
THA17    All-Vehicle Stock & KM                    THA17.20260528T000501 20    0d ago  ✓ ok
TEM12    New Vehicles by Fuel Type                 TEM12.20260527T230534 20    0d ago  ✓ ok
TEM22    Private Cars 2019-21 (archive)            TEM22.20260528T010512 20    0d ago  ✓ ok
TEM23    All Private Cars 2022+                    TEM23.20260528T000533 20    0d ago  ✓ ok
----------------------------------------------------------------------------------
Chec

## 5. External reference data (`data/external/`)

The `data/external/` folder holds small hand-curated CSVs that the prescriptive
analytics notebook (notebook 04) reads as scenario parameters and policy targets.
These are **version-controlled** (unlike `data/raw/`) because they encode
deliberate analytical choices.

### Required files

| File | Contents | Source to verify |
|---|---|---|
| `policy_targets.csv` | Official 2030 targets by KPI | Climate Action Plan 2024, NTA Strategy |
| `scenario_params.csv` | Annual growth/adoption rates for each scenario | Project assumptions (document in METHODOLOGY.md) |
| `nat_population_projections.csv` | CSO M2F2 population projection to 2030 | CSO PEP01 |

### `policy_targets.csv` — starter template

```csv
kpi,target_year,target_value,target_unit,source,notes
car_dependency_index,2030,380,cars_per_1000,"CAP 2024 / OECD","Indicative; verify against latest CAP"
pt_usage_index,2030,90,journeys_per_capita,"NTA Strategy to 2030","Assumes doubling of 2019 baseline"
ev_phev_share,2030,0.50,proportion,"CAP 2024","30% of total fleet electric → ≈50% of new cars"
ev_phev_share,2030,0.45,proportion,"Conservative estimate","Based on current adoption trajectory"
```

> **Important:** Target values are indicative. Before finalising scenario 04,
> verify against the **current** Climate Action Plan (gov.ie/climateaction) and
> the NTA Sustainable Mobility Policy. The EPA (2025) projects Ireland will
> achieve only ~23% GHG reduction by 2030 vs the 51% target — the gap itself
> is a key analytical finding for the dashboard.

In [13]:
import os
from pathlib import Path

CWD = Path.cwd()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
EXT_DIR = REPO_ROOT / "data" / "external"
EXT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_EXT = ["policy_targets.csv", "scenario_params.csv",
                "nat_population_projections.csv"]
print("data/external/ contents:")
found = list(EXT_DIR.iterdir()) if EXT_DIR.exists() else []
if found:
    for f in sorted(found):
        size = f.stat().st_size
        print(f"  {f.name:<45} {size:>8,} bytes")
else:
    print("  (empty — see starter templates above)")

print()
missing = [f for f in EXPECTED_EXT if not (EXT_DIR / f).exists()]
if missing:
    print("Files needed before running notebook 04:")
    for m in missing:
        print(f"  ⚠ {m}")
else:
    print("All required external files present.")

data/external/ contents:
  (empty — see starter templates above)

Files needed before running notebook 04:
  ⚠ policy_targets.csv
  ⚠ scenario_params.csv
  ⚠ nat_population_projections.csv


## 6. Update schedule and governance notes

| Dataset | Last significant revision | Next expected update | Action if updated |
|---|---|---|---|
| THA17 / THA18 | Annual, ~Q2 | Q2 2026 (2024 data) | Re-run notebook 01, check car dependency + intensity KPIs |
| TEM12 / TEM23 | Monthly | Each month | Re-run notebook 01 + 02 for latest fuel mix |
| THA25 | Weekly | Ongoing | Re-run notebook 01 + 02 for latest PT journeys |
| TOA11 | Monthly | Each month | Re-run notebook 01 + 02 for latest Luas numbers |
| PEA01 | Annual, ~August | August 2026 | Re-run all notebooks — population affects every per-capita KPI |
| TEM22 | Static archive | None | Do not re-download; already concatenated with TEM23 |

### What to do after a data refresh

1. Download the updated file(s) into `data/raw/` (section 2 or 3 above).
2. Run `01_data_preprocessing_and_cleaning.ipynb` → Kernel → Restart & Run All.
3. Check the `DATA_QUALITY_REPORT.md` audit log for any new warnings.
4. Re-run downstream notebooks (`02`, `03`, `04`) if KPI values change
   materially.
5. Commit the updated `data/processed/*.csv` and `docs/DATA_QUALITY_REPORT.md`.

> `data/raw/` is gitignored. Only `data/processed/` and `data/external/` are
> committed. This keeps the repository lean while ensuring the analysis-ready
> outputs are always reproducible.